# Week 11: Scope & Mini-Library — PHASE 3: Build a checkable report

*📚 Computer Programming I · ⏱️ 5 Hours · 👨‍🏫 Dr. Arif Solmaz*

## Use the same function with two sensors

We have a reusable conversion. Reuse becomes unreliable if the function silently uses whichever sensor settings were changed most recently.

Sensor A has offset 0.5 V and sensor B has offset 1.0 V. Both use gain 25 N/V. Each reports 2.5 V.

**Try this first — before code.** Calculate each force on paper. Should processing B first change the result for A? Explain what information each calculation needs.

**Why this week's tool?** Parameters pass configuration explicitly. Local variables keep each call’s work separate, so a small library can serve several sensors without hidden changes.

**By the end.** Process A, then B, then A again and explain why A’s two results agree.


<details><summary>Learning objectives</summary>

## 🎯 Learning Objectives

By the end of this week, you will be able to:

- Understand the difference between **local** and **global** scope
- Explain the **lifetime** of variables inside functions
- Use the `global` keyword (and know when to avoid it)
- Recognize and resolve **variable shadowing** issues
- Build functions that **call other functions** (composition)
- Organize code into a **mini-library** of reusable functions
- **Refactor** a flat script into well-structured functions

</details>


<details><summary>Class participation and assessment</summary>

---
## 🤝 Mechatronics Learning Contract

- **Professional relevance:** examples and core exercises model the data, sensing, automation, numerical, and decision tasks used in mechatronics engineering.
- **Interaction:** predict before running, compare reasoning with a partner, and ask whenever a step is unclear; scheduled checkpoints guarantee question time.
- **Assessment alignment:** worked examples and Core Exercises 1–8 rehearse the same reasoning operations used on exams—trace, implement, debug, interpret, and justify—while exam values and contexts may change.
- **Learning evidence:** weekly notebooks remain private practice. Non-exam evidence comes from scheduled in-class project demonstrations/presentations using a published rubric, not homework collection.

</details>


<details><summary>Class schedule and checkpoints</summary>

---
## 🧭 Five-Hour Class Roadmap

This notebook is designed for one five-hour class with four short breaks.

| Target | Activity |
|---|---|
| 00:00–00:55 | Concepts and examples → Checkpoint 1 |
| 00:55–01:05 | Break |
| 01:05–01:55 | Concepts and examples → Checkpoint 2 |
| 01:55–02:05 | Break |
| 02:05–02:55 | Concepts and examples → Checkpoint 3 |
| 02:55–03:05 | Break |
| 03:05–03:55 | Concepts and examples → Checkpoint 4 |
| 03:55–04:05 | Break |
| 04:05–04:45 | Core Practice (Exercises 1–8) → Checkpoint 5 |
| 04:45–05:00 | Review and retry failed checks |

Concept checkpoints compare your predictions with an expected answer. Checkpoint 5 is your practice reflection. No grading submission is sent by these tools. Save the notebook to keep your work. Exercises 9 and above are optional extensions.

</details>


In [1]:
# Run this setup once. These small tools give local study feedback.
_checkpoint_results = {}

def check_answer(number, answer, expected, explanation):
    actual = str(answer).strip().lower().replace(" ", "")
    target = str(expected).strip().lower().replace(" ", "")
    correct = actual == target
    _checkpoint_results[int(number)] = ("Concept check", int(correct), 1)
    if correct:
        print(f"Checkpoint {number}: correct. {explanation}")
    elif not str(answer).strip():
        print(f"Checkpoint {number}: enter your prediction, then run again.")
    else:
        print(f"Checkpoint {number}: review the example and try again.")
    return correct

def record_checkpoint(number, checks):
    """Report each concrete concept check used by the introductory notebook."""
    passed = sum(bool(correct) for _, correct in checks)
    _checkpoint_results[int(number)] = ("Concept checks", passed, len(checks))
    print(f"Checkpoint {number}: {passed}/{len(checks)} concept checks match.")
    for label, correct in checks:
        print(("OK: " if correct else "Review: ") + label)
    return passed, len(checks)

def exercise_checkpoint(number, practiced, expected=8):
    """Summarize an explicit self-report; this does not grade your code."""
    if not isinstance(practiced, (list, tuple, set)):
        _checkpoint_results.pop(int(number), None)
        print("Use a list of exercise numbers, for example [1, 2].")
        return 0, expected
    if any(type(item) is not int or not 1 <= item <= expected for item in practiced):
        _checkpoint_results.pop(int(number), None)
        print(f"Use whole exercise numbers from 1 to {expected}.")
        return 0, expected
    done = set(practiced)
    _checkpoint_results[int(number)] = ("Practice self-report", len(done), expected)
    print(f"Practice self-report: {len(done)}/{expected} core exercises reviewed.")
    print("This is your reflection, not a correctness score or a grade.")
    remaining = [str(i) for i in range(1, expected + 1) if i not in done]
    if remaining:
        print("Still to review:", ", ".join(remaining))
    print("For each exercise: test the result, explain the steps, then compare with the worked solution.")
    return len(done), expected

def show_progress_summary():
    print("\nMy study feedback (this runtime)")
    for number in range(1, 6):
        if number in _checkpoint_results:
            kind, count, total = _checkpoint_results[number]
            print(f"{number}. {kind}: {count}/{total}")
        else:
            print(f"{number}. Not run yet")
    print("These checks send no grading submission. Save your notebook to keep your work.")

print("Local study tools ready.")


Local study tools ready.


## Small prerequisite: dictionaries and two return values

A **dictionary** attaches a name (a key) to a value: `reading = {"sensor": "temp", "value": 22.5}`.
Read `reading["value"]`; update it with `reading["value"] = 23.0`; add a new entry with
`reading["unit"] = "C"`. Use `"unit" in reading` before reading an optional key.
Unlike a list index such as `row[1]`, a dictionary key describes what a value means.

`for key, value in reading.items():` visits each key/value pair. The comma **unpacks**
the pair into two local names. A function can similarly `return total, count`:
Python creates a **tuple**, an ordered collection whose positions cannot be reassigned.
`total, count = summarize(...)` unpacks its two values. A tuple may contain a mutable
list; the tuple itself does not make that list immutable.

**Türkçe:** Sözlükte değerleri isimli anahtarlarla okuruz. `return a, b` iki sonuç
döndürür; `x, y = ...` bu sonuçları sırayla iki değişkene açar. Bu küçük araçlar,
aşağıdaki öğrenci ve şekil kayıtlarını anlamak için yeterlidir.


In [2]:
reading = {"sensor": "temp", "value": 22.5}
reading["value"] = 23.0
reading["unit"] = "C"
for key, value in reading.items():
    print(key, "=", value)

def summarize(values):
    return sum(values), len(values)

total, count = summarize([20, 22, 24])
print("Total:", total, "Count:", count, "Mean:", total / count)
# Expected total 66, count 3, mean 22.0.


sensor = temp
value = 23.0
unit = C
Total: 66 Count: 3 Mean: 22.0


---
## Part 1: Review — Functions

Before we explore scope, let's quickly recap what we know about functions.

A **function** is a reusable block of code that:
- Is defined with `def`
- Can accept **parameters** (inputs)
- Can **return** a value (output)

| Concept | Syntax | Example |
|---------|--------|---------|
| Define a function | `def name(params):` | `def greet(name):` |
| Call a function | `name(args)` | `greet("Elif")` |
| Return a value | `return value` | `return total` |
| Default parameter | `def f(x=10):` | `def greet(name="Student"):` |

**Figure 1.1: A simple function with parameters and return value**

In [3]:
def calculate_average(a, b, c):
    """Calculate the average of three numbers."""
    total = a + b + c
    average = total / 3
    return average

result = calculate_average(85, 90, 78)
print(f"Average: {result:.1f}")  # Average: 84.3

Average: 84.3


**Figure 1.2: Functions with default parameters**

In [4]:
def greet(name, greeting="Hello"):
    """Greet a person with a custom or default greeting."""
    return f"{greeting}, {name}!"

print(greet("Ayse"))                  # Hello, Ayse!
print(greet("Mehmet", "Welcome"))     # Welcome, Mehmet!

Hello, Ayse!
Welcome, Mehmet!


> 💡 **Note:** Functions help us follow the **DRY principle** — Don't Repeat Yourself. If you write the same code more than once, it probably belongs in a function.

---
## Part 2: What is Scope?

**Scope** determines where a variable can be accessed in your code. Think of it like rooms in a building:

- A variable created **inside a function** is like an object in a room — it only exists in that room.
- A variable created **outside all functions** is like an object in the hallway — everyone can see it.

| Scope Type | Where Defined | Where Accessible |
|------------|---------------|------------------|
| **Local** | Inside a function | Only inside that function |
| **Global** | Outside all functions | Everywhere in the file |

**Figure 2.1: Visualizing scope**

In [5]:
# Global scope (the "hallway")
university = "ISTUN"

def show_info():
    # Local scope (a "room")
    department = "Computer Engineering"
    print(f"{department} at {university}")  # Can see both local AND global

show_info()       # Computer Engineering at ISTUN
print(university)  # ISTUN  (global — accessible here)
# print(department)  # NameError! (local — NOT accessible here)

Computer Engineering at ISTUN
ISTUN


> 💡 **Note:** Python looks for variables in this order: **Local → Enclosing → Global → Built-in** (the LEGB rule). For now, focus on Local and Global.

---
### ⏱️ Checkpoint 1 of 5 — Scope (target 00:55)

Is a variable created inside a function normally local or global?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [6]:
checkpoint_1_answer = ""  # enter your answer
check_answer(
    1, checkpoint_1_answer, 'local',
    'Function variables are local unless explicitly declared global.',
)


Checkpoint 1: enter your prediction, then run again.


False

---
## Part 3: Local Scope

Variables created **inside a function** are **local** to that function. They:
- Are **created** when the function is called
- Are **destroyed** when the function finishes
- Cannot be accessed from outside the function

**Figure 3.1: Local variables have limited lifetime**

In [7]:
def calculate_bonus(salary):
    bonus_rate = 0.15          # local variable
    bonus = salary * bonus_rate # local variable
    return bonus

result = calculate_bonus(5000)
print(f"Bonus: {result}")      # Bonus: 750.0

# These would cause NameError:
# print(bonus_rate)   # NameError: name 'bonus_rate' is not defined
# print(bonus)        # NameError: name 'bonus' is not defined

Bonus: 750.0


**Figure 3.2: Each function call creates its own local variables**

In [8]:
def count_down(start):
    counter = start   # Each call gets its OWN 'counter'
    while counter > 0:
        print(counter, end=" ")
        counter -= 1
    print("Go!")

count_down(3)   # 3 2 1 Go!
count_down(5)   # 5 4 3 2 1 Go!
# The 'counter' from the first call doesn't affect the second call

3 2 1 Go!
5 4 3 2 1 Go!


**Figure 3.3: Parameters are also local variables**

In [9]:
def double(x):
    x = x * 2   # 'x' is local — changing it doesn't affect the outside
    return x

number = 10
result = double(number)
print(f"number = {number}")  # number = 10 (unchanged!)
print(f"result = {result}")  # result = 20

number = 10
result = 20


### Names, rebinding, and mutation

A parameter is a **new local name for the object passed by the caller**. Python does
not automatically copy that object. `number = number + 1` rebinds the local name to a
new integer; `items.append(3)` mutates the same list the caller can see.
`items = [99]` then rebinds only the local name to a different list.

| Step | Caller `values` | Local `items` |
|---|---|---|
| Call function | `[1, 2]` | Same list `[1, 2]` |
| `items.append(3)` | `[1, 2, 3]` | Same list `[1, 2, 3]` |
| `items = [99]` | `[1, 2, 3]` | A different list `[99]` |

**Türkçe:** Yerel adı başka nesneye bağlamak ile ortak listedeki elemanları
değiştirmek farklı işlemlerdir. Listeyi değiştirmek çağıran kodu da etkiler.


In [10]:
def rebind_number(number):
    number = number + 1
    print("Inside number:", number)

def mutate_then_rebind(items):
    items.append(3)
    print("After append:", items)
    items = [99]
    print("After local rebinding:", items)

number = 10
rebind_number(number)
print("Caller number:", number)  # still 10
values = [1, 2]
mutate_then_rebind(values)
print("Caller list:", values)  # [1, 2, 3]


Inside number: 11
Caller number: 10
After append: [1, 2, 3]
After local rebinding: [99]
Caller list: [1, 2, 3]


---
## Part 4: Global Scope

Variables created **outside all functions** are in the **global scope**. They:
- Exist for the entire life of the program
- Can be **read** from inside any function
- Rebinding a global name inside a function requires `global`; mutating a global list or dictionary does not. Prefer passing data as parameters so the dependency is visible.

**Figure 4.1: Reading global variables inside functions**

In [11]:
PI = 3.14159
GRAVITY = 9.81

def circle_area(radius):
    return PI * radius ** 2   # Reading global PI — OK!

def fall_distance(time):
    return 0.5 * GRAVITY * time ** 2   # Reading global GRAVITY — OK!

print(f"Circle area: {circle_area(5):.2f}")     # Circle area: 78.54
print(f"Fall distance: {fall_distance(3):.2f}")  # Fall distance: 44.15

Circle area: 78.54
Fall distance: 44.15


**Figure 4.2: Trying to modify a global variable (this creates a NEW local variable!)**

In [12]:
score = 100   # global

def add_points():
    score = 150   # This creates a NEW local variable, doesn't change the global!
    print(f"Inside function: score = {score}")   # 150

add_points()
print(f"Outside function: score = {score}")      # 100 (unchanged!)

Inside function: score = 150
Outside function: score = 100


> 💡 **Note:** When you assign a value to a variable inside a function, Python always creates a **local** variable — even if a global variable with the same name exists. This is a common source of confusion!

---
### ⏱️ Checkpoint 2 of 5 — Globals (target 01:55)

For reusable functions, should global state usually be avoided? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [13]:
checkpoint_2_answer = ""  # enter your answer
check_answer(
    2, checkpoint_2_answer, 'yes',
    'Explicit inputs and outputs are easier to test.',
)


Checkpoint 2: enter your prediction, then run again.


False

---
## Part 5: The `global` Keyword

The `global` keyword tells Python: "I want to use the **global** variable, not create a new local one."

**Figure 5.1: Using the `global` keyword**

In [14]:
total_score = 0   # global

def add_score(points):
    global total_score      # Tell Python: use the global variable
    total_score += points

add_score(10)
add_score(25)
add_score(15)
print(f"Total: {total_score}")   # Total: 50

Total: 50


**Figure 5.2: Why `global` is usually a bad idea**

In [15]:
# BAD — uses global keyword, hard to track
counter = 0
def increment_bad():
    global counter
    counter += 1

# GOOD — uses parameters and return values instead
def increment_good(counter):
    return counter + 1

# The "good" version is easier to understand and test:
my_counter = 0
my_counter = increment_good(my_counter)   # 1
my_counter = increment_good(my_counter)   # 2
print(f"Counter: {my_counter}")           # Counter: 2

Counter: 2


> 💡 **Note:** Avoid `global` in most cases. Instead, pass values as **parameters** and use **return** to send results back. This makes your code easier to understand, test, and debug. Global **constants** (like `PI` or `GRAVITY`) are fine because they never change.

---
## Part 6: Variable Shadowing

**Variable shadowing** occurs when a local variable has the **same name** as a global variable. The local variable "shadows" (hides) the global one inside the function.

**Figure 6.1: Shadowing in action**

In [16]:
message = "Hello from global!"   # global variable

def show_message():
    message = "Hello from local!"   # shadows the global 'message'
    print(message)                  # prints the LOCAL version

show_message()       # Hello from local!
print(message)       # Hello from global!  (global is unchanged)

Hello from local!
Hello from global!


**Figure 6.2: A tricky scope error — UnboundLocalError**

In [17]:
value = 10

def tricky():
    # Python sees 'value = ...' below, so it treats 'value' as LOCAL
    # But we try to READ it before assigning — error!
    try:
        print(value)     # UnboundLocalError!
        value = 20       # This line makes Python think 'value' is local
    except UnboundLocalError:
        print("Error: tried to read local variable before assigning it!")

tricky()

Error: tried to read local variable before assigning it!


> 💡 **Note:** If Python sees an assignment to a variable **anywhere** in a function, it treats that variable as local for the **entire** function — even lines before the assignment. This is a common gotcha!

---
## Part 7: Functions Calling Functions

Functions can call other functions! This is called **composition** — building complex behavior from simple building blocks.

**Figure 7.1: Simple function composition**

In [18]:
def square(x):
    return x ** 2

def add(a, b):
    return a + b

def sum_of_squares(a, b):
    """Calculate a² + b² using the functions above."""
    return add(square(a), square(b))   # Calling two other functions!

print(sum_of_squares(3, 4))   # 25  (9 + 16)

25


---
### ⏱️ Checkpoint 3 of 5 — Composition (target 02:55)

What do we call one function using another function?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [19]:
checkpoint_3_answer = ""  # enter your answer
check_answer(
    3, checkpoint_3_answer, 'composition',
    'Function composition builds larger behavior from small units.',
)


Checkpoint 3: enter your prediction, then run again.


False

**Figure 7.2: Building a student report with composition**

In [20]:
def calculate_average(grades):
    """Calculate the average of a list of grades."""
    return sum(grades) / len(grades)

def determine_letter(average):
    """Convert numeric average to letter grade."""
    if average >= 90: return "AA"
    elif average >= 80: return "BA"
    elif average >= 70: return "BB"
    elif average >= 60: return "CB"
    elif average >= 50: return "CC"
    else: return "FF"

def pass_or_fail(letter):
    """Determine if the student passed."""
    return "PASS" if letter != "FF" else "FAIL"

def generate_report(name, grades):
    """Generate a full report — uses all three functions above."""
    avg = calculate_average(grades)
    letter = determine_letter(avg)
    status = pass_or_fail(letter)
    return f"{name}: {avg:.1f} ({letter}) - {status}"

# Use the composed function:
print(generate_report("Elif Kaya", [85, 92, 78, 90]))
# Elif Kaya: 86.2 (BA) - PASS

print(generate_report("Burak Demir", [40, 35, 45, 38]))
# Burak Demir: 39.5 (FF) - FAIL

Elif Kaya: 86.2 (BA) - PASS
Burak Demir: 39.5 (FF) - FAIL


> 💡 **Note:** Each function does **one thing well**. Then we combine them to build more complex behavior. This makes code easier to test, debug, and reuse.

---
## Part 8: Organizing Code with Functions — The Mini-Library Approach

A **mini-library** is a collection of related functions that work together. Think of it as a toolbox for a specific task.

Good library design principles:

| Principle | Description |
|-----------|-------------|
| **Single Responsibility** | Each function does one thing |
| **Clear Naming** | Function name describes what it does |
| **Docstrings** | Each function has a description |
| **Consistent Interface** | Similar functions work in similar ways |
| **Explicit Effects** | Prefer returned results; document intentional printing, file writing, or mutation of inputs |

**Figure 8.1: A mini-library for text analysis**

In [21]:
# ═══════════════════════════════════════════
# TEXT ANALYSIS MINI-LIBRARY
# ═══════════════════════════════════════════

def count_words(text):
    """Count the number of words in a text."""
    return len(text.split())

def count_sentences(text):
    """Count sentences (ending with . ! or ?)."""
    count = 0
    for char in text:
        if char in ".!?":
            count += 1
    return max(count, 1)   # At least 1 sentence

def average_word_length(text):
    """Calculate the average length of words."""
    words = text.split()
    total_length = sum(len(word) for word in words)
    return total_length / len(words) if words else 0

def text_summary(text):
    """Generate a summary using all helper functions."""
    words = count_words(text)
    sentences = count_sentences(text)
    avg_len = average_word_length(text)
    return (
        f"Words: {words}\n"
        f"Sentences: {sentences}\n"
        f"Avg word length: {avg_len:.1f}\n"
        f"Words per sentence: {words / sentences:.1f}"
    )

# Test the library:
sample = "Python is great. It is easy to learn! Do you agree?"
print(text_summary(sample))

Words: 11
Sentences: 3
Avg word length: 3.7
Words per sentence: 3.7


---
## Part 9: Refactoring — From Script to Functions

**Refactoring** means restructuring code without changing what it does. Let's take a flat script and break it into functions.

**Figure 9.1: BEFORE refactoring — everything in one block**

In [22]:
# ❌ BEFORE: A messy script — everything mixed together
students = [
    {"name": "Zeynep", "grades": [85, 90, 78]},
    {"name": "Omer", "grades": [45, 50, 42]},
    {"name": "Selin", "grades": [92, 88, 95]},
]

for s in students:
    total = 0
    for g in s["grades"]:
        total += g
    avg = total / len(s["grades"])
    if avg >= 90: letter = "AA"
    elif avg >= 80: letter = "BA"
    elif avg >= 70: letter = "BB"
    elif avg >= 60: letter = "CB"
    elif avg >= 50: letter = "CC"
    else: letter = "FF"
    if letter != "FF": status = "PASS"
    else: status = "FAIL"
    print(f"{s['name']}: {avg:.1f} ({letter}) - {status}")

Zeynep: 84.3 (BA) - PASS
Omer: 45.7 (FF) - FAIL
Selin: 91.7 (AA) - PASS


**Figure 9.2: AFTER refactoring — clean, reusable functions**

In [23]:
# ✅ AFTER: Clean functions — each does one thing

def calc_average(grades):
    """Calculate average of a list of grades."""
    return sum(grades) / len(grades)

def get_letter_grade(average):
    """Convert numeric average to letter grade."""
    if average >= 90: return "AA"
    elif average >= 80: return "BA"
    elif average >= 70: return "BB"
    elif average >= 60: return "CB"
    elif average >= 50: return "CC"
    else: return "FF"

def get_status(letter):
    """Determine pass/fail from letter grade."""
    return "PASS" if letter != "FF" else "FAIL"

def print_report(students):
    """Print grade report for all students."""
    for s in students:
        avg = calc_average(s["grades"])
        letter = get_letter_grade(avg)
        status = get_status(letter)
        print(f"{s['name']}: {avg:.1f} ({letter}) - {status}")

# --- Main Program ---
students = [
    {"name": "Zeynep", "grades": [85, 90, 78]},
    {"name": "Omer", "grades": [45, 50, 42]},
    {"name": "Selin", "grades": [92, 88, 95]},
]

print_report(students)

Zeynep: 84.3 (BA) - PASS
Omer: 45.7 (FF) - FAIL
Selin: 91.7 (AA) - PASS


> 💡 **Note:** After refactoring, each function can be tested individually. Need to change how letter grades work? Just edit `get_letter_grade()` — nothing else needs to change.

---
## Part 10: Putting It Together — A Complete Mini-Library

Let's build a **BMI (Body Mass Index) Calculator Library** that demonstrates everything we've learned: scope, composition, and organization.

**Figure 10.1: Complete BMI Calculator mini-library**

In [24]:
# ═══════════════════════════════════════════
# BMI CALCULATOR MINI-LIBRARY
# ═══════════════════════════════════════════

# --- Constants (global, but that's OK for constants) ---
BMI_UNDERWEIGHT = 18.5
BMI_NORMAL = 25.0
BMI_OVERWEIGHT = 30.0

# --- Helper Functions ---
def calculate_bmi(weight_kg, height_m):
    """Calculate BMI from weight (kg) and height (meters)."""
    return weight_kg / (height_m ** 2)

def bmi_category(bmi):
    """Return the BMI category as a string."""
    if bmi < BMI_UNDERWEIGHT:
        return "Underweight"
    elif bmi < BMI_NORMAL:
        return "Normal weight"
    elif bmi < BMI_OVERWEIGHT:
        return "Overweight"
    else:
        return "Obese"

def cm_to_meters(cm):
    """Convert centimeters to meters."""
    return cm / 100

def bmi_report(name, weight_kg, height_cm):
    """Generate a complete BMI report."""
    height_m = cm_to_meters(height_cm)   # helper function
    bmi = calculate_bmi(weight_kg, height_m)   # helper function
    category = bmi_category(bmi)   # helper function
    return (
        f"Patient: {name}\n"
        f"Weight: {weight_kg} kg, Height: {height_cm} cm\n"
        f"BMI: {bmi:.1f}\n"
        f"Category: {category}"
    )

# --- Main Program ---
print(bmi_report("Ahmet Yilmaz", 75, 178))
print()
print(bmi_report("Fatma Celik", 58, 165))

Patient: Ahmet Yilmaz
Weight: 75 kg, Height: 178 cm
BMI: 23.7
Category: Normal weight

Patient: Fatma Celik
Weight: 58 kg, Height: 165 cm
BMI: 21.3
Category: Normal weight


---
### ⏱️ Checkpoint 4 of 5 — Shadowing (target 03:55)

Can a local variable have the same name as a global variable? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [25]:
checkpoint_4_answer = ""  # enter your answer
check_answer(
    4, checkpoint_4_answer, 'yes',
    'The local name shadows the global name inside the function.',
)


Checkpoint 4: enter your prediction, then run again.


False

---
## Exercises — Problems to Solve

> Each exercise is a problem. Think about the **STEPS** before you code. Decompose the problem, plan your approach, then implement it.

Complete the exercises below. Each exercise cell starts with `# ✏️ [EXn]` — **do not remove this line**.

### Core Practice and Optional Extension

- **Exercises 1–8:** core in-class practice.
- **Exercises 9 and above:** optional extension; these are not homework.
- At Checkpoint 5, list the exercises you have tested and can explain. This is a self-report, not automatic grading.


---
## Exercises

Complete the exercises below. Each exercise cell starts with `# ✏️ [EXn]` — **do not remove this line**.

### Exercise 1: Scope Prediction (Easy)

Without running the code first, **predict** what the following program will print. Then add `print()` statements in the cell below to show the output.

```python
x = 10

def foo():
    x = 20
    print(x)

def bar():
    print(x)

foo()
bar()
print(x)
```

**Expected output:**
```
20
10
10
```

<details><summary>💡 Hint</summary>

`foo()` creates a local `x = 20`, which shadows the global `x = 10`. `bar()` reads the global `x` because it has no local `x`. After both calls, the global `x` is still `10`.

</details>

In [26]:
# ✏️ [EX1]
# Type the code from above and verify your prediction.
# Add a comment explaining WHY each print gives its value.


### Exercise 2: Fix the Scope Bug (Easy)

The following code has a **scope bug**. Find and fix it so that `get_full_greeting()` works correctly.

```python
def set_greeting():
    greeting = "Merhaba"

def get_full_greeting(name):
    return f"{greeting}, {name}!"

set_greeting()
print(get_full_greeting("Deniz"))
```

**Expected output:**
```
Merhaba, Deniz!
```

<details><summary>💡 Hint</summary>

The variable `greeting` is local to `set_greeting()`. Fix it by making `set_greeting()` return the greeting, or pass the greeting as a parameter to `get_full_greeting()`.

</details>

In [27]:
# ✏️ [EX2]
# Fix the scope bug. Do NOT use the 'global' keyword.


### Exercise 3: Temperature Library (Medium)

Create a temperature conversion mini-library with these functions:

| Function | Formula |
|----------|---------|
| `c_to_f(celsius)` | F = C × 9/5 + 32 |
| `f_to_c(fahrenheit)` | C = (F − 32) × 5/9 |
| `c_to_k(celsius)` | K = C + 273.15 |
| `k_to_c(kelvin)` | C = K − 273.15 |

**Expected output:**
```
100°C = 212.0°F
32°F = 0.0°C
0°C = 273.15K
300K = 26.85°C
```

<details><summary>💡 Hint</summary>

Each function takes one number and returns one number. Use the formulas directly. Test each function with known values (e.g., 100°C = 212°F, 0°C = 273.15K).

</details>

In [28]:
# ✏️ [EX3]
# Create your temperature conversion library below.


### Exercise 4: Math Helper Library (Medium)

Create a math helper library with these functions:

- `is_prime(n)` — returns `True` if `n` is prime, `False` otherwise
- `factorial(n)` — returns n! (e.g., factorial(5) = 120)
- `gcd(a, b)` — returns the greatest common divisor
- `lcm(a, b)` — returns the least common multiple (use `gcd` inside!)

**Expected output:**
```
is_prime(7) = True
is_prime(10) = False
factorial(5) = 120
gcd(12, 8) = 4
lcm(12, 8) = 24
```

<details><summary>💡 Hint</summary>

For `is_prime`: check divisibility from 2 to the square root of n. For `gcd`: use the Euclidean algorithm (while b != 0: a, b = b, a % b). For `lcm`: use the formula `lcm(a,b) = abs(a*b) // gcd(a,b)`.

</details>

In [29]:
# ✏️ [EX4]
# Create your math helper library below.


### Exercise 5: String Helper Library (Medium)

Create a string helper library with these functions:

- `is_palindrome(text)` — returns `True` if text reads the same forwards and backwards (ignore case and spaces)
- `count_vowels(text)` — returns the count of vowels (a, e, i, o, u)
- `reverse_string(text)` — returns the reversed text

**Expected output:**
```
is_palindrome('racecar') = True
is_palindrome('hello') = False
count_vowels('Hello World') = 3
reverse_string('Python') = nohtyP
```

<details><summary>💡 Hint</summary>

For `is_palindrome`: convert to lowercase, remove spaces, then compare with reversed version. For `count_vowels`: loop through each character and check if it's in `"aeiouAEIOU"`. For `reverse_string`: use slicing `text[::-1]`.

</details>

In [30]:
# ✏️ [EX5]
# Create your string helper library below.


### Exercise 6: Statistics Library (Medium)

Create a statistics library with:

- `mean(numbers)` — returns the average
- `median(numbers)` — returns the middle value (sort first!)
- `mode(numbers)` — returns the most frequent value

**Expected output:**
```
Data: [4, 7, 2, 7, 3, 7, 5]
Mean: 5.0
Median: 5
Mode: 7
```

<details><summary>💡 Hint</summary>

For `median`: sort the list, then return the middle element (if odd length) or average of the two middle elements (if even length). For `mode`: count occurrences of each number using a dictionary or loop, then return the one with the highest count.

</details>

**Input contract:** use a nonempty list of numbers. For tied modes, return the value that appears first in the original list. Do not mutate the original list while finding the median. The bridge below deliberately explores an empty-list error.


In [31]:
# ✏️ [EX6]
# Create your statistics library below.


### Exercise 7: Geometry Library (Medium)

Create a geometry library with:

- `circle_area(radius)` — returns π × r² (use `PI = 3.14159` as a global constant)
- `rect_area(width, height)` — returns width × height
- `triangle_area(base, height)` — returns 0.5 × base × height

Also create a `shape_report(shapes)` function that takes a list of shape dictionaries and prints a report.

**Expected output:**
```
circle (r=5): area = 78.54
rectangle (4x6): area = 24.00
triangle (b=3, h=8): area = 12.00
```

<details><summary>💡 Hint</summary>

Define `PI = 3.14159` as a global constant. Each shape dictionary could have a "type" key and the relevant dimensions. The `shape_report` function checks the type and calls the appropriate area function.

</details>

In [32]:
# ✏️ [EX7]
# Create your geometry library below.


### Exercise 8: Refactor — Inline to Functions (Medium)

Refactor the following inline code into well-organized functions. The output should remain **exactly the same**.

```python
# Inline code to refactor:
prices = [120, 250, 80, 310, 45, 190]
total = 0
discounted_total = 0
for p in prices:
    total += p
    if p > 200:
        discounted_total += p * 0.9
    elif p > 100:
        discounted_total += p * 0.95
    else:
        discounted_total += p
print(f"Original total: {total} TL")
print(f"Discounted total: {discounted_total:.2f} TL")
print(f"You saved: {total - discounted_total:.2f} TL")
```

**Expected output:**
```
Original total: 995 TL
Discounted total: 923.50 TL
You saved: 71.50 TL
```

<details><summary>💡 Hint</summary>

Create functions like `apply_discount(price)`, `calculate_total(prices)`, `calculate_discounted_total(prices)`, and `print_summary(prices)`. Each function should do one thing.

</details>

In [33]:
# ✏️ [EX8]
# Refactor the inline code into functions below.


---
### Checkpoint 5 of 5 — Practice reflection (target 04:45)

After Exercises 1–8, edit `practiced_exercises` in the next cell.
List only the exercise numbers whose results you have tested and whose steps you can explain.
Leave the list empty until you have done that work. This is your explicit self-report;
the tool does not inspect or grade your solution and does not count execution history.

**Türkçe:** Bu liste öz değerlendirmedir. Hücreyi çalıştırmak tek başına yeterli değildir;
sonucu kontrol et ve çözüm adımlarını açıklayabildiğinden emin ol.


In [34]:
# Add an exercise number only after testing and explaining your own work.
# Example: [1, 2] records your reflection about Exercises 1 and 2.
# Türkçe: Bu liste öz değerlendirmedir; kodunuzun doğruluğunu otomatik ölçmez.
practiced_exercises = []
exercise_checkpoint(5, practiced_exercises, expected=8)
show_progress_summary()


Practice self-report: 0/8 core exercises reviewed.
This is your reflection, not a correctness score or a grade.
Still to review: 1, 2, 3, 4, 5, 6, 7, 8
For each exercise: test the result, explain the steps, then compare with the worked solution.

My study feedback (this runtime)
1. Concept check: 0/1
2. Concept check: 0/1
3. Concept check: 0/1
4. Concept check: 0/1
5. Practice self-report: 0/8
These checks send no grading submission. Save your notebook to keep your work.


---
## 🌟 Optional Extension

Exercises 9 and above are optional enrichment. Stop here if the five-hour class has ended.


### Exercise 9: Validator Library (Medium)

Create a validator library with:

- `validate_age(age)` — returns `True` if age is between 0 and 150
- `validate_email(email)` — returns `True` if email contains `@` and `.` after `@`
- `validate_id(student_id)` — returns `True` if student_id is exactly 10 digits

Also create `validate_student(name, age, email, student_id)` that checks all three and returns a list of error messages (empty list if all valid).

**Expected output:**
```
Valid student: []
Invalid student: ['Invalid age', 'Invalid email']
```

<details><summary>💡 Hint</summary>

For `validate_email`: check if `"@"` is in the email, and if there's a `"."` after the `"@"`. For `validate_id`: check if `len(student_id) == 10` and `student_id.isdigit()`. The combined function should call each validator and collect error messages.

</details>

In [35]:
# ✏️ [EX9]
# Create your validator library below.


### Exercise 10: Composition (Medium)

Build a `process_grade(raw_score, bonus, curve)` function that:

1. Calls `apply_bonus(score, bonus)` — adds bonus points
2. Calls `apply_curve(score, curve)` — adds curve points
3. Calls `clamp(score, 0, 100)` — ensures score is between 0 and 100
4. Calls `to_letter(score)` — converts to letter grade

**Expected output:**
```
Raw: 72, Bonus: 5, Curve: 8 -> Score: 85, Grade: BA
Raw: 95, Bonus: 10, Curve: 5 -> Score: 100, Grade: AA
Raw: 30, Bonus: 0, Curve: 5 -> Score: 35, Grade: FF
```

<details><summary>💡 Hint</summary>

`clamp(value, min_val, max_val)` returns `min_val` if value < min_val, `max_val` if value > max_val, otherwise value. Use `min()` and `max()` built-in functions: `max(min_val, min(value, max_val))`.

</details>

**Exercise grading scale:** AA ≥90, BA ≥80, BB ≥70, CB ≥60, CC ≥50, otherwise FF; only FF fails. These are rules for this programming exercise, not the course assessment weights.


In [36]:
# ✏️ [EX10]
# Build your composed grade processing function below.


### Exercise 11: Grade Report Generator (Challenge)

Build a complete grade report system that combines multiple helper functions. Given a list of students with their exam scores, generate a formatted report.

Requirements:
- Calculate weighted average (Midterm: 40%, Final: 60%)
- Determine letter grade and pass/fail status
- Calculate class statistics (highest, lowest, class average)
- Print a formatted report

**Test data:**
```python
students = [
    {"name": "Ayse Yildiz", "midterm": 78, "final": 85},
    {"name": "Kemal Ozturk", "midterm": 92, "final": 88},
    {"name": "Deniz Arslan", "midterm": 45, "final": 40},
    {"name": "Hakan Cetin", "midterm": 65, "final": 72},
    {"name": "Sibel Kara", "midterm": 88, "final": 95},
]
```

**Expected output (approximate):**
```
═══════════════════════════════════════
         GRADE REPORT
═══════════════════════════════════════
Ayse Yildiz     : 82.2 (BA) - PASS
Kemal Ozturk    : 89.6 (BA) - PASS
Deniz Arslan    : 42.0 (FF) - FAIL
Hakan Cetin     : 69.2 (CB) - PASS
Sibel Kara      : 92.2 (AA) - PASS
═══════════════════════════════════════
Class Average: 75.0
Highest: 92.2 (Sibel Kara)
Lowest: 42.0 (Deniz Arslan)
Pass Rate: 80.0%
═══════════════════════════════════════
```

<details><summary>💡 Hint</summary>

Break it into functions: `weighted_average(midterm, final)`, `letter_grade(avg)`, `pass_fail(letter)`, `class_stats(student_results)`, `print_report(students)`. Build from the bottom up.

</details>

**Exercise grading scale:** AA ≥90, BA ≥80, BB ≥70, CB ≥60, CC ≥50, otherwise FF; only FF fails. These are rules for this programming exercise, not the course assessment weights.


In [37]:
# ✏️ [EX11]
# Build your grade report generator below.


### Exercise 12: Physics Calculator (Challenge)

Create these small functions, then compose them in `physics_report(mass, distance, time, height)`.

| Function | Formula | Meaning |
|---|---|---|
| `velocity(distance, time)` | `distance / time` | **Average** speed for straight forward motion |
| `acceleration(v_final, v_initial, time)` | `(v_final - v_initial) / time` | Constant acceleration |
| `force(mass, acceleration)` | `mass * acceleration` | Net force along the motion |
| `kinetic_energy(mass, velocity)` | `0.5 * mass * velocity**2` | Energy at the stated instantaneous speed |
| `potential_energy(mass, height)` | `mass * 9.81 * height` | Gravitational energy relative to zero height |
| `momentum(mass, velocity)` | `mass * velocity` | Momentum along the motion |

**Physical model:** a 10 kg cart moves in a straight line from rest with constant
acceleration, covering 100 m in 5 s. Its height is 20 m above the chosen reference.
There is no reversal. Time is positive and mass is non-negative. The force requested
is the net force **along the motion**; height is only used to report gravitational energy.

The average speed is `100/5 = 20 m/s`. It is not the final speed.
From `distance = (v_initial + v_final)*time/2`, with `v_initial = 0`, obtain
`v_final = 2*distance/time = 40 m/s`, then `a = (40-0)/5 = 8 m/s²`.
Use the final speed for final kinetic energy and momentum.

**Expected output:**
```text
Average speed: 20.00 m/s
Final velocity: 40.00 m/s
Acceleration: 8.00 m/s²
Net force: 80.00 N
Final kinetic energy: 8000.00 J
Potential energy: 1962.00 J
Final momentum: 400.00 kg·m/s
```

**Türkçe:** `yol/zaman` ortalama sürattir. Durarak başlayıp sabit ivmeyle hızlanan
cisimde son sürat bunun iki katıdır. Kuvvet, enerji ve momentumda aynı fiziksel
modeli kullanın. Fonksiyonları kısa tutun; rapor fonksiyonu bunları sırayla çağırır.


In [38]:
# ✏️ [EX12]
# Create your physics calculator library below.


### Bridge Exercise: Preview of Error Handling

Next week, we'll learn about **error handling** with `try/except`. For now, try this: what happens when you call your physics calculator with `time = 0`? What about your statistics library with an empty list?

Write code that **demonstrates these errors** happening. We'll learn how to handle them gracefully next week!

**Expected behavior:**
```
velocity(100, 0) causes: ZeroDivisionError
mean([]) causes: ZeroDivisionError
```

<details><summary>💡 Hint</summary>

Just call the functions with invalid inputs and observe the errors. You can use `try/except` as a preview: `try: velocity(100, 0)` and `except ZeroDivisionError as e: print(e)`.

</details>

In [39]:
# ✏️ [EXBridge]
# Experiment with calling functions with invalid inputs.
# What errors do you get? Can you predict them?


## Worked solutions and study support

Try each problem first. Then compare your reasoning and test cases with the complete
[Week 11 worked solutions](../solutions/Week_11_Solutions.ipynb).
The solution notebook includes every core exercise, optional exercise, and this week's bridge when present.

If you are using Colab, [open the published solution notebook](https://colab.research.google.com/github/ArifSolmaz/courses/blob/main/fall/cp1/solutions/Week_11_Solutions.ipynb).
Open it in a separate runtime. Running a solution first should not supply hidden variables to your own work.

**Türkçe:** Önce kendi çözümünü dene. Sonra adımları ve testleri karşılaştır; çözümü kapatıp farklı bir örneği kendin çöz.

[Simple course guide](../STUDY_GUIDE.md) · [All worked solutions](../solutions/README.md)
